# Azure Event Grid Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/azure_event_grid/event_grid_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/azure_event_grid/event_grid_demo.ipynb)

## Business Scenario

Azure services emit events when files arrive or devices publish telemetry. You need to ingest those events and validate them immediately.

## Value Proposition

- Event-driven ingestion with schema validation
- Consistent processing for Azure-native events
- Clean handoff to downstream analytics

---

## Goals

1. Connect to Event Grid
2. Validate incoming events
3. Write clean events to storage


##  Step 1: Setup Azure Event Grid

Before running this notebook, you need:
1. An Azure Event Grid topic or system topic (e.g., Storage Account events)
2. An Event Grid subscription
3. The endpoint URL and subscription name

Set your environment variable:
```bash
export AZURE_EVENT_GRID_ENDPOINT="https://your-topic.region.eventgrid.azure.net/api/events"
```

##  Step 2: Review the Contract

Our contract defines the expected Event Grid schema and quality rules.

In [ ]:
from pathlib import Path
from lakelogic import DataProcessor

BASE = Path.cwd()
contract_path = BASE / "event_grid_contract.yaml"
if not contract_path.exists():
    candidate = BASE / "examples" / "03_data_sources" / "streaming" / "azure_event_grid" / "event_grid_contract.yaml"
    if candidate.exists():
        BASE = candidate.parent
        contract_path = candidate

RUN_LIVE = False  # Set True when your streaming infrastructure is available

sample_events = [{'id': 'evt-001', 'eventType': 'Microsoft.Storage.BlobCreated', 'subject': '/blobServices/default/containers/test/blobs/file1.txt', 'eventTime': '2026-02-15T10:10:00', 'data': {'size': 1234}}, {'id': 'evt-002', 'eventType': 'Microsoft.Storage.BlobDeleted', 'subject': '/blobServices/default/containers/test/blobs/file2.txt', 'eventTime': '2026-02-15T10:12:00', 'data': {'size': 0}}]

if RUN_LIVE:
    print("Live streaming is disabled by default in this demo.")
    print("Set RUN_LIVE = True and configure credentials/endpoints to stream.")
else:
    processor = DataProcessor(contract=contract_path)
    result = processor.run(sample_events, source_path="sample")
    print(result)
    print(f"Good: {len(result.good)} | Bad: {len(result.bad)}")


##  Step 3: Start the Event Grid Listener

This will connect to your Event Grid subscription and start processing events.

##  Step 4: Trigger Test Events

Upload a file to your Azure Storage Account to trigger a `BlobCreated` event.

You should see the event appear in the LakeLogic logs and get materialized to Delta Lake.

##  Summary

You just:
-  Connected to Azure Event Grid
-  Validated storage events against a contract
-  Materialized events to Delta Lake

This pattern enables **event-driven data pipelines** where your lakehouse automatically reacts to infrastructure changes!